# Retry failed ablation cases

Notebook này retry **chỉ** các case có `status: failed` trong một ablation run đã có, sau đó tạo file predictions đã gộp. Artifact gốc không bị ghi đè.

Trước khi chạy, bảo đảm source trong `/kaggle/working/TextMining` đã có backoff cho lỗi 429 và các biến `LLM_API_KEY`, `LLM_BASE_URL`, `LLM_BASE_MODEL` (hoặc model đã được resolve trong run gốc) đã được cấu hình.

In [1]:
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/PhuongThao-2005/TextMining.git'
# Use a dedicated directory so an older TextMining clone cannot shadow the runner.
REPO_DIR = Path('/kaggle/working/TextMining_retry_runner_main')

if not (REPO_DIR / '.git').is_dir():
    if REPO_DIR.exists():
        raise RuntimeError(f'{REPO_DIR} exists but is not a Git repository; inspect it before cloning.')
    subprocess.run(
        ['git', 'clone', '--branch', 'main', '--single-branch', REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print(f'Using existing repository: {REPO_DIR}')

commit = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'],
    check=True, capture_output=True, text=True,
).stdout.strip()
branch = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'branch', '--show-current'],
    check=True, capture_output=True, text=True,
).stdout.strip()
if branch != 'main':
    raise RuntimeError(f'Expected branch main, found {branch!r} in {REPO_DIR}')
runner_path = REPO_DIR / 'scripts' / 'run_ablation_config.py'
if not runner_path.is_file():
    raise FileNotFoundError(
        f'Runner is missing at {runner_path}. Confirm that branch main contains the ablation runner.'
    )
print('Repository branch:', branch)
print('Repository commit:', commit)
print('Runner:', runner_path)


Cloning into '/kaggle/working/TextMining_retry_runner_main'...


Repository branch: main
Repository commit: d834975d1f296080a6c36df41525fa1ecbd710f9
Runner: /kaggle/working/TextMining_retry_runner_main/scripts/run_ablation_config.py


In [2]:
# Dán đường dẫn run DeepSeek gốc vào đây.
ORIGINAL_RUN_DIR = Path('/kaggle/working/TextMining_retry_runner_main/e2e_LLM_Reasoning/deepseekv3.1-thinking-CoT')
RUNS_ROOT = Path('/kaggle/working/evaluation_runs/ablation')
RETRY_RUN_ID = 'deepseek_retry_failed_cases'

# Chỉ dùng None nếu muốn retry toàn bộ failed cases.
EXPECTED_FAILED_CASES = 7


In [3]:
import json
import sys
from collections import Counter

if not REPO_DIR.is_dir():
    raise FileNotFoundError(f'Repository not found: {REPO_DIR}')
if 'PASTE_DEEPSEEK_RUN_DIRECTORY_HERE' in str(ORIGINAL_RUN_DIR):
    raise ValueError('Set ORIGINAL_RUN_DIR to the completed DeepSeek run directory first.')
if not ORIGINAL_RUN_DIR.is_dir():
    raise FileNotFoundError(f'Original run not found: {ORIGINAL_RUN_DIR}')

sys.path.insert(0, str(REPO_DIR))
# Remove a cached scripts package from an older repository clone.
for module_name in tuple(sys.modules):
    if module_name == 'scripts' or module_name.startswith('scripts.'):
        sys.modules.pop(module_name, None)

required = ['manifest.json', 'resolved_config.yaml', 'e2e_predictions.jsonl', 'errors.jsonl']
missing = [name for name in required if not (ORIGINAL_RUN_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing original artifacts: {missing}')

manifest = json.loads((ORIGINAL_RUN_DIR / 'manifest.json').read_text(encoding='utf-8'))
print('Config:', manifest.get('config_name'))
print('Original run:', ORIGINAL_RUN_DIR)


Config: LLM-CoTReasoning
Original run: /kaggle/working/TextMining_retry_runner_main/e2e_LLM_Reasoning/deepseekv3.1-thinking-CoT


In [4]:
def read_jsonl(path: Path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def case_id(row: dict):
    return row.get('qa_id') or row.get('case_id') or row.get('id')

base_predictions = read_jsonl(ORIGINAL_RUN_DIR / 'e2e_predictions.jsonl')
errors = read_jsonl(ORIGINAL_RUN_DIR / 'errors.jsonl')
failed_ids = {case_id(row) for row in base_predictions if row.get('status') == 'failed'}
failed_ids.discard(None)

print('Total prediction rows:', len(base_predictions))
print('Failed prediction rows:', len(failed_ids))
print('Error groups:')
for (stage, exception_type, retryable), count in Counter(
    (row.get('stage'), row.get('exception_type'), row.get('retryable')) for row in errors
).most_common():
    print(f'  {count:>3}  stage={stage}; type={exception_type}; retryable={retryable}')

if not failed_ids:
    raise RuntimeError('No failed predictions found; nothing to retry.')
if EXPECTED_FAILED_CASES is not None and len(failed_ids) != EXPECTED_FAILED_CASES:
    raise RuntimeError(f'Expected {EXPECTED_FAILED_CASES} failed cases, found {len(failed_ids)}. Check ORIGINAL_RUN_DIR.')

print('Failed IDs:', sorted(failed_ids))


Total prediction rows: 500
Failed prediction rows: 7
Error groups:
    7  stage=generation; type=RuntimeError; retryable=False
Failed IDs: ['vlrag-0cc7e882be', 'vlrag-71f687ab53', 'vlrag-8ad537015f', 'vlrag-d6259b812b', 'vlrag-f0d0601ecb', 'vlrag-nuance-14497a648c', 'vlrag-nuance-bce9060852']


In [5]:
import yaml

resolved_config = yaml.safe_load((ORIGINAL_RUN_DIR / 'resolved_config.yaml').read_text(encoding='utf-8'))
qa_source = Path(str(resolved_config['benchmark']['path']))
if not qa_source.is_file():
    raise FileNotFoundError(f'Benchmark from resolved config is unavailable: {qa_source}')

qa_rows = read_jsonl(qa_source)
retry_rows = [row for row in qa_rows if case_id(row) in failed_ids]
if len(retry_rows) != len(failed_ids):
    found = {case_id(row) for row in retry_rows}
    raise RuntimeError(f'Could not find every failed case in benchmark. Missing: {sorted(failed_ids - found)}')

retry_qa_path = Path('/kaggle/working') / f'{RETRY_RUN_ID}.jsonl'
retry_qa_path.write_text(
    '\n'.join(json.dumps(row, ensure_ascii=False) for row in retry_rows) + '\n',
    encoding='utf-8',
)

# Preserve every resolved setting from the original run; change only benchmark.path.
resolved_config['benchmark']['path'] = str(retry_qa_path)
print(f'Created retry benchmark with {len(retry_rows)} cases: {retry_qa_path}')


Created retry benchmark with 7 cases: /kaggle/working/deepseek_retry_failed_cases.jsonl


In [6]:
from scripts.run_ablation_config import run_ablation_config

retry_outcome = run_ablation_config(
    str(manifest['config_name']),
    config_file=REPO_DIR / 'configs' / 'ablation_configs.yaml',
    output_root=RUNS_ROOT,
    run_id=RETRY_RUN_ID,
    limit=None,
    dry_run=False,
    project_root=REPO_DIR,
    resolved_config_override=resolved_config,
)

print('Retry status:', retry_outcome.status)
print('Retry output:', retry_outcome.output_dir)
if retry_outcome.status != 'completed':
    raise RuntimeError(f'Retry did not complete: {retry_outcome.error or retry_outcome.status}')


AblationConfigError: Required path validation failed: retrieval.dense.index_path=/kaggle/working/e2e_runtime_inputs/faiss

In [ ]:
from datetime import datetime, timezone
import shutil

from evaluation.e2e_runner import (
    E2ERunResult, METRIC_KEYS, aggregate_agent_metrics, aggregate_latency, write_e2e_artifacts,
)
from evaluation.metrics import aggregate, aggregate_by
from generation.citations import aggregate_citation_metrics

retry_run_dir = Path(retry_outcome.output_dir)
retry_predictions = read_jsonl(retry_run_dir / 'e2e_predictions.jsonl')
retry_by_id = {case_id(row): row for row in retry_predictions}
unrecovered = [item for item in sorted(failed_ids) if retry_by_id.get(item, {}).get('status') != 'success']
if unrecovered:
    raise RuntimeError(f'Retry still failed for {len(unrecovered)} case(s): {unrecovered}')

merged_predictions = [
    retry_by_id[case_id(row)] if case_id(row) in failed_ids else row
    for row in base_predictions
]
assert len(merged_predictions) == len(base_predictions)
assert all(row.get('status') == 'success' for row in merged_predictions), 'Some rows are not successful.'

# Recompute every aggregate from the complete 500-row prediction set.
metric_keys = list(METRIC_KEYS)
if merged_predictions and 'judge_correctness' in merged_predictions[0]:
    metric_keys.extend(['judge_correctness', 'judge_faithfulness', 'judge_answer_relevancy'])
counts = {
    'total_input': len(merged_predictions),
    'successful': len(merged_predictions),
    'failed': 0,
    'skipped': 0,
    'evaluated': len(merged_predictions),
}
original_metrics = json.loads((ORIGINAL_RUN_DIR / 'e2e_metrics.json').read_text(encoding='utf-8'))
metrics = {
    'qa_path': original_metrics.get('qa_path'),
    'config': original_metrics.get('config') or resolved_config,
    'counts': counts,
    'metric_denominator': 'Per-metric: averages exclude successful cases where that metric is not applicable.',
    'overall': aggregate(merged_predictions, metric_keys),
    'by_category': aggregate_by(merged_predictions, 'category', metric_keys),
    'by_answer_type': aggregate_by(merged_predictions, 'answer_type', metric_keys),
    'by_difficulty': aggregate_by(merged_predictions, 'difficulty', metric_keys),
    'metric_keys': metric_keys,
    'agent_metrics': aggregate_agent_metrics(merged_predictions),
    'citation_metrics': aggregate_citation_metrics(merged_predictions),
}
latency = aggregate_latency(merged_predictions)
latency['denominator'] = 'Each stage uses cases with a recorded value for that stage.'
latency['counts'] = counts

repaired_dir = Path('/kaggle/working/repaired_ablation_runs') / f"{manifest['run_id']}_repaired"
repaired_dir.mkdir(parents=True, exist_ok=False)
result = E2ERunResult(predictions=merged_predictions, errors=[], metrics=metrics, latency=latency)
artifact_paths = write_e2e_artifacts(repaired_dir, result, report_name='report.md')
shutil.copy2(ORIGINAL_RUN_DIR / 'resolved_config.yaml', repaired_dir / 'resolved_config.yaml')

repaired_manifest = dict(manifest)
repaired_manifest.update({
    'run_id': f"{manifest['run_id']}_repaired",
    'status': 'completed',
    'end_time': datetime.now(timezone.utc).isoformat(),
    'output_directory': str(repaired_dir),
    'output_artifacts': {
        **artifact_paths,
        'manifest': str(repaired_dir / 'manifest.json'),
        'resolved_config': str(repaired_dir / 'resolved_config.yaml'),
        'readme': str(repaired_dir / 'README.md'),
    },
    'completed_case_count': len(merged_predictions),
    'failed_case_count': 0,
    'skipped_case_count': 0,
    'evaluated_case_count': len(merged_predictions),
    'error_summary': None,
    'repair': {
        'original_run': str(ORIGINAL_RUN_DIR),
        'retry_run': str(retry_run_dir),
        'recovered_case_ids': sorted(failed_ids),
    },
})
(repaired_dir / 'manifest.json').write_text(
    json.dumps(repaired_manifest, ensure_ascii=False, indent=2), encoding='utf-8'
)

overall = metrics['overall']
readme_lines = [
    '# Repaired Ablation Result', '',
    f"- Original run: `{ORIGINAL_RUN_DIR}`",
    f"- Retry run: `{retry_run_dir}`",
    f"- Recovered cases: {len(failed_ids)}",
    f"- Final predictions: {len(merged_predictions)} successful, 0 failed, 0 skipped", '',
    '## Updated metrics', '',
    '| Metric | Value |', '| --- | ---: |',
]
for key in metric_keys:
    value = overall.get(key)
    readme_lines.append(f'| {key} | {float(value):.4f} |' if value is not None else f'| {key} | — |')
readme_lines.extend(['', 'Metrics were recomputed from the repaired complete prediction set.'])
(repaired_dir / 'README.md').write_text('\n'.join(readme_lines) + '\n', encoding='utf-8')

print('Repaired run:', repaired_dir)
print('Predictions:', repaired_dir / 'e2e_predictions.jsonl')
print('Metrics:', repaired_dir / 'e2e_metrics.json')
print('README:', repaired_dir / 'README.md')
print('Counts:', counts)
